# 07. 온도 CSV 22개 센서의 zone별 상관계수와 히트맵

**작성자: 정수진** **Issue:** [#115](https://github.com/posco-3bungi-pdm/pdm-project/issues/115)  
**브랜치:** `feat/115-zone-correlation-jinny`

- 입력: 원본 `T-CR1-CAL01_온도.csv`의 센서 22개 전체 (`CP-Z4M` 포함)
- 계산에서 빼는 컬럼: 시각 `MEAS_DT`, LOT 번호 `LOT_NO`
- 순서: CLN → Z1~Z9 → COOLING. 표와 그림에 같은 순서를 사용한다.
- 계산: `pandas.corr()`의 Pearson 상관계수. 같은 행의 관측값끼리 비교한다.

## 1. 경로와 라이브러리

저장소 루트 또는 `notebooks/jinny/`에서 실행할 수 있다.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
FIG_DIR = os.path.join(ROOT, "figures", "jinny", "07_zone_correlation")
os.makedirs(FIG_DIR, exist_ok=True)

## 2. 원본 읽기

기본값 `LOT_NO = None`은 전체 LOT다. 특정 LOT만 보려면 예를 들어 `LOT_NO = 240058`로 바꾼다.
정기수리 전후 구간도 포함한 원본이며, 별도 제외나 보간은 하지 않는다.

In [ ]:
LOT_NO = None

df = pd.read_csv(TEMP_CSV, encoding="utf-8-sig")
if LOT_NO is not None:
    df = df[df["LOT_NO"] == LOT_NO]
if df.empty:
    raise ValueError("데이터가 없습니다. LOT_NO를 확인하세요.")

print(f"관측 {len(df):,}행 / LOT {df['LOT_NO'].nunique()}개")

## 3. zone 순서로 센서 22개 정렬

온도 CSV에 들어 있는 TC·OP·CP·CLN 컬럼을 모두 포함한다.

In [ ]:
ZONES = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4", "CP-Z4M"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

sensors = []
for columns in ZONES.values():
    sensors += columns

print("센서:", len(sensors), "개")

## 4. 22×22 상관계수 표

- +1에 가까우면 함께 오르내리고, −1에 가까우면 반대로 움직인다.
- 0 근처는 선형 관계가 약하다는 뜻이다.
- 결측값은 각 센서 쌍에서만 제외한다. 유효한 쌍이 2개 미만이거나 값이 일정한 센서는 `NaN`이다.

In [ ]:
corr = df[sensors].corr(method="pearson", min_periods=2)

with pd.option_context("display.max_rows", 22, "display.max_columns", 22):
    display(corr.round(2))

## 5. 22×22 히트맵

흰 선은 zone 경계다. 계산할 수 없는 값은 회색으로 표시한다.
전체 LOT 그림은 `22_sensor_correlation_all_lots.png`, 특정 LOT 그림은 LOT 번호가 붙은 이름으로 저장한다.

In [ ]:
scope = "all_lots" if LOT_NO is None else f"lot_{LOT_NO}"
fig, ax = plt.subplots(figsize=(16, 14))
ax.set_facecolor("#dddddd")
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, center=0, square=True,
    annot_kws={"size": 8}, cbar_kws={"label": "Pearson r", "shrink": 0.6}, ax=ax,
)

boundary = 0
for columns in ZONES.values():
    boundary += len(columns)
    ax.axhline(boundary, color="white", linewidth=1.5)
    ax.axvline(boundary, color="white", linewidth=1.5)

ax.set_title("소둔로 센서 간 상관관계 분석", fontsize=20, pad=20)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

figure_path = os.path.join(FIG_DIR, f"22_sensor_소둔로 센서 간 상관관계 분석.png")
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("그림 저장:", figure_path)

## 해석할 때

전체 LOT를 합치면 LOT별 온도 수준이나 공통 조업 변화도 상관계수에 섞인다.
필요하면 2번 셀의 `LOT_NO`를 바꿔 같은 LOT 안에서도 확인한다.
이 결과는 같은 시점의 선형 관계이며, 센서 간 시간 지연이나 인과관계·고장 여부를 뜻하지 않는다.

원본 CSV와 실행 출력은 커밋하지 않는다. 공유할 그림은 확인 후 별도로 추가한다.